In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401 (needed for 3D)

In [ ]:
def plot_partial_dependence_iLOCO(
    X, mp_ensemble, j1, j2,
    grid_resolution=30,
    grid_percentiles=(0.05, 0.95),
    kind="surface",                # "surface" or "contour"
    classifier=False,
    class_index=1,                 # for predict_proba: which class probability to show
    sample_ids=None,               # optionally subset rows to average over (array-like of indices)
    show=True,
    cmap="viridis",
    progress=False
):
    """
    Plot 2D partial dependence surface for features j1 and j2 using a minipatch ensemble.

    Parameters
    ----------
    X : numpy.ndarray or pandas.DataFrame
        Feature matrix. If DataFrame, column names are preserved for labels.
    mp_ensemble : dict
        Output of mp_ensemble(). Must contain keys:
          - "ensemble": list of fitted models
          - "mp_features": (M, B) boolean array of features used by each model
    j1, j2 : int or str
        Feature indices (ints) or column names (if X is DataFrame).
    grid_resolution : int
        Number of points per axis in the grid (grid_resolution x grid_resolution).
    grid_percentiles : tuple(float,float)
        Percentile range from which to build grid (avoid extreme tails).
    kind : str
        "surface" (3D surface) or "contour" (2D contour + heatmap).
    classifier : bool
        If True, will call model.predict_proba and use `class_index`. If False, uses model.predict.
    class_index : int
        Which column of predict_proba to use (binary: 1 for positive class).
    sample_ids : array-like or None
        If provided, average predictions only over these sample rows. Otherwise average over all rows.
    show : bool
        If True, show the matplotlib figure before returning.
    cmap : str
        Colormap for the surface/contour.
    progress : bool
        If True, show a progress bar (requires tqdm).

    Returns
    -------
    X_grid, Y_grid, Z : np.ndarray
        Meshgrid arrays and PD surface values (shape = (grid_resolution, grid_resolution)).
    """

    # --- handle pandas input and feature names ---
    if isinstance(X, pd.DataFrame):
        feature_names = X.columns.tolist()
        X_np = X.to_numpy()
        # allow j1, j2 to be column names
        if isinstance(j1, str):
            j1 = feature_names.index(j1)
        if isinstance(j2, str):
            j2 = feature_names.index(j2)
    else:
        X_np = np.asarray(X)
        M = X_np.shape[1]
        feature_names = [f"feature_{i}" for i in range(M)]

    N, M = X_np.shape
    if sample_ids is None:
        sample_ids = np.arange(N)
    else:
        sample_ids = np.asarray(sample_ids)

    ensemble = mp_ensemble["ensemble"]
    mp_features = mp_ensemble["mp_features"]  # shape (M, B) boolean
    B = len(ensemble)

    # --- build grid from data percentiles (avoids extreme outliers) ---
    vals_j1 = np.nanpercentile(X_np[:, j1], np.linspace(100 * grid_percentiles[0], 100 * grid_percentiles[1], grid_resolution))
    vals_j2 = np.nanpercentile(X_np[:, j2], np.linspace(100 * grid_percentiles[0], 100 * grid_percentiles[1], grid_resolution))
    Xg, Yg = np.meshgrid(vals_j1, vals_j2)   # shape (grid_res, grid_res)

    # Z will store PD value at each grid point
    Z = np.zeros_like(Xg, dtype=float)

    # optionally iterate with progress bar
    outer_iter = range(grid_resolution * grid_resolution)
    if progress:
        outer_iter = trange(grid_resolution * grid_resolution, desc="PD grid")

    # Pre-allocate temp arrays to avoid repeated allocation inside loops
    X_temp = X_np.copy()  # will be overwritten for each grid point

    # For each grid point, set X_temp[:, j1] and X_temp[:, j2] to the grid values,
    # then predict with each model using only the model's feature columns.
    idx = 0
    for k in outer_iter:
        i = k // grid_resolution
        j = k % grid_resolution
        g1 = Xg[i, j]
        g2 = Yg[i, j]

        # replace j1,j2 across all samples (we'll average only over sample_ids)
        # copy only the columns being changed to avoid copying whole matrix repeatedly
        # (but we will restore them afterwards)
        orig_j1 = X_temp[:, j1].copy()
        orig_j2 = X_temp[:, j2].copy()
        X_temp[:, j1] = g1
        X_temp[:, j2] = g2

        # accumulate predictions across models
        # We'll average across models (simple mean) then across samples (PD standard)
        model_preds = np.zeros((B, sample_ids.shape[0]), dtype=float)

        for b, model in enumerate(ensemble):
            # get the features used by this model (indices)
            idx_F_b = np.where(mp_features[:, b])[0]
            # build design matrix for this model
            X_in = X_temp[np.ix_(sample_ids, idx_F_b)]
            # safe predict: classifier -> predict_proba if available else predict
            if classifier and hasattr(model, "predict_proba"):
                proba = model.predict_proba(X_in)
                # If multiclass, ensure class_index exists
                if proba.ndim == 2 and proba.shape[1] > class_index:
                    preds_b = proba[:, class_index]
                else:
                    # fallback: use first column if indexing fails
                    preds_b = proba[:, 0]
            else:
                preds_b = model.predict(X_in)
                # if predict returns shape (n, k) (rare), try to handle
                if preds_b.ndim > 1:
                    # take first column as fallback
                    preds_b = preds_b[:, 0]

            model_preds[b, :] = preds_b

        # average across models then across samples -> single scalar PD value
        mean_across_models = np.nanmean(model_preds, axis=0)   # shape (n_samples_sub)
        pd_value = np.nanmean(mean_across_models)             # scalar

        Z[i, j] = pd_value

        # restore original columns to X_temp
        X_temp[:, j1] = orig_j1
        X_temp[:, j2] = orig_j2

        idx += 1

    # --- Plotting ---
    fig = plt.figure(figsize=(10, 7))
    if kind == "surface":
        ax = fig.add_subplot(111, projection="3d")
        surf = ax.plot_surface(Xg, Yg, Z, rstride=1, cstride=1, cmap=cmap, edgecolor='none', linewidth=0, antialiased=True)
        ax.set_xlabel(feature_names[j1])
        ax.set_ylabel(feature_names[j2])
        ax.set_zlabel("f̂ (partial dependence)")
        ax.set_title(f"Partial dependence: {feature_names[j1]} vs {feature_names[j2]}")
        fig.colorbar(surf, shrink=0.6, aspect=10, label="f̂")
    elif kind == "contour":
        ax = fig.add_subplot(111)
        cs = ax.contourf(Xg, Yg, Z, cmap=cmap)
        ax.set_xlabel(feature_names[j1])
        ax.set_ylabel(feature_names[j2])
        ax.set_title(f"Partial dependence (contour): {feature_names[j1]} vs {feature_names[j2]}")
        fig.colorbar(cs, label="f̂")
    else:
        raise ValueError("kind must be 'surface' or 'contour'")

    if show:
        plt.show()

    return Xg, Yg, Z